# An Interpreter for a Simple Programming Language

F# counterpart of `Python/ANTLR/Interpreter.ipynb`. The missing Python/ANTLR generated dependencies are replaced by the compiled FsLexYacc parser in Chapter 07. Parsing, tree drawing and execution have separate owners. The interpreter itself is defined visibly in Chapter 07 and reused here; its implementation is reproduced below for reading, while `#load` imports the canonical notebook export.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "../Chapter-07/LanguageSyntax/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "../Chapter-07/LanguageSyntax/bin/Debug/net9.0/LanguageSyntax.dll"
#load "../Support/Visualization.fsx"
#load "../Chapter-07/Interpreter.fsx"
open LanguageSyntax
open Interpreter
open Visualization
open System.IO

## Grammar and typed syntax

The `.fsy` productions return `Expr` and `Statement` unions. These constructors replace open-ended nested tuples and make missing cases visible during compilation. The simple interpreter handles assignment, printing, conditionals and loops; the worked frame extends it with functions and returns.

```fsharp
%{
open LanguageSyntax
%}
%token <LanguageSyntax.Value> NUMBER
%token <string> IDENTIFIER STRING
%token IF WHILE FOR FUNCTION RETURN
%token PLUS MINUS TIMES DIVIDE MODULO ASSIGN EQ NE LT GT LE GE AND OR NOT
%token LPAREN RPAREN LBRACE RBRACE SEMICOLON COMMA EOF
%start program
%type <LanguageSyntax.Statement> program
%%
program: statements EOF { Block $1 }
statements: { [] } | nonemptyStatements { $1 }
nonemptyStatements: statement { [$1] } | statement nonemptyStatements { $1 :: $2 }
statement: IF LPAREN expr RPAREN statement { If($3, $5) }
         | WHILE LPAREN expr RPAREN statement { While($3, $5) }
         | FOR LPAREN assign SEMICOLON expr SEMICOLON assign RPAREN statement { For($3, $5, $7, $9) }
         | FUNCTION IDENTIFIER LPAREN identifiers RPAREN statement { Define($2, $4, $6) }
         | RETURN expr SEMICOLON { Return $2 }
         | LBRACE statements RBRACE { Block $2 }
         | assign SEMICOLON { $1 }
         | expr SEMICOLON { Expression $1 }
assign: IDENTIFIER ASSIGN expr { Assign($1, $3) }
identifiers: { [] } | identifierList { $1 }
identifierList: IDENTIFIER { [$1] } | identifierList COMMA IDENTIFIER { $1 @ [$3] }
expr: expr OR conjunction { Binary(Or, $1, $3) } | conjunction { $1 }
conjunction: conjunction AND comparison { Binary(And, $1, $3) } | comparison { $1 }
comparison: sum EQ sum { Binary(Equal, $1, $3) }
          | sum NE sum { Binary(NotEqual, $1, $3) }
          | sum LT sum { Binary(Less, $1, $3) }
          | sum GT sum { Binary(Greater, $1, $3) }
          | sum LE sum { Binary(LessEqual, $1, $3) }
          | sum GE sum { Binary(GreaterEqual, $1, $3) }
          | sum { $1 }
sum: sum PLUS product { Binary(Add, $1, $3) }
   | sum MINUS product { Binary(Subtract, $1, $3) }
   | product { $1 }
product: product TIMES factor { Binary(Multiply, $1, $3) }
       | product DIVIDE factor { Binary(Divide, $1, $3) }
       | product MODULO factor { Binary(Modulo, $1, $3) }
       | factor { $1 }
factor: LPAREN expr RPAREN { $2 }
      | NUMBER { Literal $1 }
      | STRING { Literal(Text $1) }
      | IDENTIFIER { Variable $1 }
      | IDENTIFIER LPAREN expressions RPAREN { Call($1, $3) }
      | NOT factor { Not $2 }
      | MINUS factor { Negative $2 }
expressions: { [] } | expressionList { $1 }
expressionList: expr { [$1] } | expressionList COMMA expr { $1 @ [$3] }
```


## Interpreter operations

The I/O record is the real boundary between computation and user interaction. A deterministic `Read` function makes examples reproducible; an output collector lets us verify printed values. The same interpreter can be used with console functions.

Integers remain exact throughout the variable environment and IO boundary; `/` returns a real value and `%` follows Python’s floor-quotient rule. The numeric comparison helpers are also reused by the extended interpreter.

```fsharp
let divide (numerator: float) (denominator: float) =
    if denominator = 0.0 then failwith "Division by zero"
    numerator / denominator

open System.Numerics

let number = function
    | (Integer _ | Real _) as value -> value
    | value -> failwithf "Expected a number, got %A" value
let asNumber = function
    | Integer n -> float n
    | Real n -> n
    | value -> failwithf "Expected a number, got %A" value

// Compare the integer with the actual represented real, without rounding it first.
let compareIntegerReal (integer: BigInteger) (real: float) =
    if System.Double.IsNaN real then failwith "NaN cannot be ordered"
    elif System.Double.IsPositiveInfinity real then -1
    elif System.Double.IsNegativeInfinity real then 1
    else
        let whole = BigInteger real
        let result = compare integer whole
        if result <> 0 then result else compare (float integer) real
let compareValues left right =
    match left, right with
    | Integer a, Integer b -> compare a b
    | Integer a, Real b -> compareIntegerReal a b
    | Real a, Integer b -> -(compareIntegerReal b a)
    | Real a, Real b -> compare a b
    | Text a, Text b -> System.String.CompareOrdinal(a, b)
    | Boolean a, Boolean b -> compare a b
    | _ -> failwithf "Cannot compare %A and %A" left right
let equalValues left right =
    match left, right with
    | Real a, Real b -> a = b
    | Integer _, Real b when System.Double.IsNaN b -> false
    | Real a, Integer _ when System.Double.IsNaN a -> false
    | (Integer _ | Real _), (Integer _ | Real _) -> compareValues left right = 0
    | _ -> left = right

// .NET integer division truncates; Python's integer modulo uses a floor quotient.
let floorDivide (left: BigInteger) (right: BigInteger) =
    if right.IsZero then failwith "Division by zero"
    let quotient, remainder = left / right, left % right
    if not remainder.IsZero && left.Sign <> right.Sign then quotient - BigInteger.One else quotient

// Python modulo has the divisor's sign, including an exact zero remainder.
let floatModulo (left: float) (right: float) =
    if right = 0.0 then failwith "Modulo by zero"
    let remainder = left % right
    if remainder = 0.0 then System.Math.CopySign(0.0, right)
    elif (remainder < 0.0) <> (right < 0.0) then remainder + right
    else remainder

// Round the exact ratio once, rather than rounding each large operand to float.
let divideIntegers (numerator: BigInteger) (denominator: BigInteger) =
    if denominator.IsZero then failwith "Division by zero"
    let sign = if (numerator.Sign < 0) <> (denominator.Sign < 0) then -1.0 else 1.0
    let a, b = BigInteger.Abs numerator, BigInteger.Abs denominator
    if a.IsZero then sign * 0.0
    else
        let candidate = a.GetBitLength() - b.GetBitLength()
        if candidate > 1024L then failwith "Integer division result is too large for a real value"
        elif candidate < -1075L then sign * 0.0
        else
            let power = int candidate
            let belowPower =
                if power >= 0 then a < (b <<< power)
                else (a <<< -power) < b
            let exponent = if belowPower then power - 1 else power
            // Normal doubles have 53 significant bits; subnormals use units of 2^-1074.
            let scale = if exponent < -1022 then 1074 else 52 - exponent
            let scaledA, scaledB =
                if scale >= 0 then a <<< scale, b
                else a, b <<< -scale
            let quotient, remainder = scaledA / scaledB, scaledA % scaledB
            let twiceRemainder = remainder * 2I
            let rounded =
                if twiceRemainder > scaledB || (twiceRemainder = scaledB && not quotient.IsEven)
                then quotient + 1I else quotient
            let result = System.Math.ScaleB(float rounded, -scale)
            if System.Double.IsInfinity result then failwith "Integer division result is too large for a real value"
            sign * result

let arithmetic op left right =
    match left, right with
    | Integer a, Integer b ->
        match op with
        | Add -> Integer(a + b) | Subtract -> Integer(a - b) | Multiply -> Integer(a * b)
        | Divide -> Real(divideIntegers a b)
        | Modulo -> Integer(a - floorDivide a b * b)
        | _ -> failwith "Expected an arithmetic operator"
    | _ ->
        let a, b = asNumber left, asNumber right
        match op with
        | Add -> Real(a + b) | Subtract -> Real(a - b) | Multiply -> Real(a * b)
        | Divide -> Real(divide a b) | Modulo -> Real(floatModulo a b)
        | _ -> failwith "Expected an arithmetic operator"

type IO = { Read: unit -> Value; Write: Value -> unit }

let rec evaluate io values = function
    | Literal value -> number value
    | Variable name ->
        match Map.tryFind name values with
        | Some value -> value
        | None -> failwithf "Undefined variable: %s" name
    | Call("read", []) -> number (io.Read())
    | Call("print", [expr]) -> io.Write(evaluate io values expr); Integer 0I
    | Negative expr ->
        match evaluate io values expr with
        | Integer n -> Integer(-n) | Real n -> Real(-n)
        | _ -> failwith "Unary minus expects a number"
    | Binary(op, left, right) ->
        arithmetic op (evaluate io values left) (evaluate io values right)
    | _ -> failwith "Expression is outside the simple language"

let evaluateBool io values = function
    | Binary(op, left, right) ->
        let l, r = evaluate io values left, evaluate io values right
        match op with
        | Equal -> equalValues l r | NotEqual -> not (equalValues l r)
        | Less -> compareValues l r < 0 | Greater -> compareValues l r > 0
        | LessEqual -> compareValues l r <= 0 | GreaterEqual -> compareValues l r >= 0
        | _ -> failwith "Expected a comparison"
    | _ -> failwith "Expected a comparison"

let rec execute io values = function
    | Block statements -> List.fold (execute io) values statements
    | Assign(name, expr) -> Map.add name (evaluate io values expr) values
    | Expression expr -> evaluate io values expr |> ignore; values
    | If(test, body) -> if evaluateBool io values test then execute io values body else values
    | While(test, body) ->
        let rec repeat current =
            if evaluateBool io current test then repeat (execute io current body) else current
        repeat values
    | _ -> failwith "Statement is outside the simple language"

let run io source = execute io Map.empty (Parse.program source)
let main file =
    let read () =
        printf "Please enter a natural number: "
        Integer(BigInteger.Parse(System.Console.ReadLine(), System.Globalization.CultureInfo.InvariantCulture))
    let io = { Read = read; Write = (valueText >> printfn "%s") }
    run io (System.IO.File.ReadAllText file)
```


In [ ]:
let executeTuple io values statements =
    statements |> List.fold (execute io) values

let output = ResizeArray<Value>()
let io = { Read = (fun () -> Integer 10I); Write = output.Add }
let runFile file =
    let source = File.ReadAllText file
    let statements = LanguageSyntax.Parse.program source
    printfn "%A" statements
    let values = execute io Map.empty statements
    values, statementTree statements |> treeToDot

let sumValues, sumDot = runFile "sum.sl"
checkNotebook (List.ofSeq output = [Integer 5050I])
checkNotebook (sumValues["i"] = Integer 100I)
output.Clear()
let factorialValues, factorialDot = runFile "factorial.sl"
checkNotebook (List.ofSeq output = [Integer 3628800I])
checkNotebook (factorialValues["f"] = Integer 3628800I)
output.Clear()
let exactIO = { io with Read = (fun () -> Integer 9007199254740993I) }
let exactValues = run exactIO "x := read(); print(x + 1); q := 3 / 2; r := -7 % 3;"
checkNotebook (List.ofSeq output = [Integer 9007199254740994I])
checkNotebook (exactValues["q"] = Real 1.5)
checkNotebook (exactValues["r"] = Integer 2I)


In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(renderDot sumDot), "text/html") |> ignore

## Optional user input

`Interpreter.main "sum.sl"` uses the console boundary. It is deliberately not called during a full notebook run. Generated files are ignored build products; no notebook cell deletes Python, Java, image or other user files.